# Migrate IceChunk repos onto the time grid

One-off, in place: puts repos written before the fixed time grid onto it so `append`/`upsert` ingests can write to them. No data is read or rewritten. Per repo it:

- checks every group's time axis is sorted, unique, on the source's grid, and starts at the source's first day;
- records `append_dim`, `time_origin`, `time_step`, `configuration_name` and `timeseries_type` on the root group;
- adds `created_at`/`updated_at` (the last commit's time, the same in every group, so no stage reprocesses) and `source_last_modified` (from the source listing).

Existing gaps stay gaps. A repo that fails a check is left alone and its problems are printed. Reversible: it is one icechunk commit.

Existing pyramid levels keep their zarr fill value (0): their snow-free chunks were never stored and read as 0 only through it. Rebuild a domain's pyramids if you want unwritten slots to read as missing there too.

**Credentials** come from the environment, never from this notebook:
- `AWS_ACCESS_KEY_ID`/`AWS_SECRET_ACCESS_KEY`/`AWS_SESSION_TOKEN` with write access to the repo bucket (e.g. `aws configure export-credentials --profile fved-aws-poweruser --format env`);
- `M3WORKS_AWS_ACCESS_KEY_ID`/`M3WORKS_AWS_SECRET_ACCESS_KEY` for the iSnobal listing.

In [ ]:
import sys
from pathlib import Path

import icechunk as ic

# The flow code imports as Prefect runs it: workflows.* and utils.*
PREFECT_WORKFLOWS = Path.cwd().parents[2] / "prefect-workflows"
sys.path[:0] = [str(PREFECT_WORKFLOWS), str(PREFECT_WORKFLOWS / "workflows" / "ingests")]

from utils import grid_utils as gu
from utils.time_grid import migrate_repo
from workflows.models.gridded_sources import ISnobal

Set the repo location and the domains to migrate.

In [ ]:
DEST_BUCKET = "dev-fved-icechunk-warehouse-rti-use1"
BASE_PREFIX = "icechunk-ingests"
DOMAINS = ["boise"]  # start with one, then add the rest


def open_repo(source):
    storage = gu.build_icechunk_s3_storage(DEST_BUCKET, f"{BASE_PREFIX}/{source.repository_name()}", from_env=True)
    return ic.Repository.open(storage)

Dry run: checks only, writes nothing.

In [ ]:
for domain in DOMAINS:
    source = ISnobal(domain=domain)
    report = migrate_repo(open_repo(source), source, source.repository_name(), dry_run=True)
    print(domain, report)

Migrate. Afterwards, an `append` ingest over an already stored window should report no changed steps in any stage.

In [ ]:
for domain in DOMAINS:
    source = ISnobal(domain=domain)
    report = migrate_repo(open_repo(source), source, source.repository_name(), dry_run=False)
    print(domain, report)